# DOCTOR v3.4 — Colab 부팅 노트북

Google Drive 캐시(`doctor-cache/doctrine_chroma_db.zip`)에서 ChromaDB를 복원하고,
Colab GPU 위에 Ollama(`qwen3:8b`) + FastAPI 백엔드를 띄운 뒤 ngrok 으로 공개 URL을 발급합니다.

**전제 조건**
- Colab Runtime → **GPU** 선택 (T4 이상 권장)
- 본인 Google 계정 Drive의 `MyDrive/doctor-cache/doctrine_chroma_db.zip` 업로드 완료
- ngrok 무료 계정의 authtoken 준비 (셀 6에서 입력)

셀을 위에서 아래로 차례대로 실행하세요.

## 1. Drive 마운트 + ChromaDB 압축 해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

CACHE_DIR = '/content/drive/MyDrive/doctor-cache'
CHROMA_ZIP = f'{CACHE_DIR}/doctrine_chroma_db.zip'
CHROMA_DIR = '/content/chroma_db'

if not os.path.exists(f'{CHROMA_DIR}/chroma.sqlite3'):
    print('ChromaDB 압축 해제 중...')
    with zipfile.ZipFile(CHROMA_ZIP, 'r') as z:
        z.extractall(CHROMA_DIR)
    print(f'OK ChromaDB 완료: {CHROMA_DIR}')
else:
    print('OK ChromaDB 이미 존재, 스킵')

## 2. Ollama 설치 + qwen3:8b 다운로드

In [ ]:
import subprocess, time

print('Ollama 설치 중...')
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

print('Ollama 서버 시작...')
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

print('qwen3:8b 다운로드 중... (~5GB 근처, 5~15분 환경 의존)')
subprocess.run(['ollama', 'pull', 'qwen3:8b'], check=True)
print('OK Ollama 완료')

## 3. 백엔드 패키지 설치

In [ ]:
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'fastapi', 'uvicorn', 'chromadb==0.6.0',
    'sentence-transformers', 'pyngrok', 'httpx',
    'python-dotenv', 'pandas'
], check=True)
print('OK 패키지 설치 완료')

## 4. 저장소 clone + ChromaDB 연결

In [ ]:
import subprocess, os, shutil

repo_dir = '/content/doctrine-agent-local'

if not os.path.exists(repo_dir):
    subprocess.run([
        'git', 'clone',
        'https://github.com/psj950101-dev/doctrine-agent-local.git',
        repo_dir
    ], check=True)
    print('OK 저장소 clone 완료')
else:
    subprocess.run(['git', '-C', repo_dir, 'pull'])
    print('OK 저장소 최신화')

# ChromaDB를 backend/chroma_db 로 복사 (config.py: CHROMA_DIR = backend/{CHROMA_DIR})
chroma_dest = f'{repo_dir}/backend/chroma_db'
os.makedirs(chroma_dest, exist_ok=True)

if not os.path.exists(f'{chroma_dest}/chroma.sqlite3'):
    print('ChromaDB 복사 중...')
    shutil.copytree(CHROMA_DIR, chroma_dest, dirs_exist_ok=True)
    print('OK ChromaDB 연결 완료')
else:
    print('OK ChromaDB 이미 연결됨')

## 5. `.env` 작성 + FastAPI 서버 시작

Colab 환경에서는 백엔드와 Ollama 가 같은 VM 에 있으므로 `OLLAMA_BASE_URL=http://localhost:11434` 로 직결.
공군 컬렉션 이름이 ChromaDB(`airforce_doctrine`) vs 백엔드 기본값(`air_force_doctrine`) 으로 다르므로
`CHROMA_COLLECTION_AIR_FORCE` 환경변수로 오버라이드합니다.

In [ ]:
import subprocess, time

env_content = '''# Colab 부팅 전용 .env (Ollama·백엔드가 같은 VM 에 있음)
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=qwen3:8b
OLLAMA_TIMEOUT=180
OLLAMA_MAX_TOKENS=512

EMBEDDING_MODEL=BAAI/bge-m3
EMBEDDING_BATCH_SIZE=8

CHROMA_DIR=chroma_db
# Drive 캐시 zip 의 공군 컬렉션 이름과 백엔드 기본값이 달라서 오버라이드
CHROMA_COLLECTION_ARMY=army_doctrine
CHROMA_COLLECTION_NAVY=navy_doctrine
CHROMA_COLLECTION_AIR_FORCE=airforce_doctrine

TOP_K_MAX=20
RAG_CHUNK_CHAR_LIMIT=1200
RAG_CONTEXT_CHAR_LIMIT=4500
RETRIEVAL_MAX_DISTANCE=0.75

CORS_ORIGINS=*
LOG_LEVEL=INFO
'''

# config.py 가 PROJECT_ROOT/.env 를 먼저 읽으므로 repo 루트에 작성
env_path = f'{repo_dir}/.env'
with open(env_path, 'w') as f:
    f.write(env_content)
print(f'OK .env 작성 완료: {env_path}')

server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=f'{repo_dir}/backend'
)
time.sleep(8)
print('OK FastAPI 서버 시작 (인제스트는 백그라운드로 진행, /health 즉시 응답)')

## 6. ngrok 터널 발급 + 헬스체크

**박성준 본인의 ngrok authtoken** 을 `NGROK_TOKEN` 에 직접 입력하세요.
토큰은 https://dashboard.ngrok.com/get-started/your-authtoken 에서 발급합니다.

In [ ]:
from pyngrok import ngrok
import requests, time

# 박성준 ngrok authtoken 직접 입력 (GitHub 에 커밋 금지)
NGROK_TOKEN = 'YOUR_NGROK_TOKEN'
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000)
base_url = public_url.public_url
print(f'\n[BACKEND URL] {base_url}\n')

time.sleep(3)
try:
    r = requests.get(f'{base_url}/health', timeout=10)
    print(f'OK 헬스체크: {r.json()}')
except Exception as e:
    print(f'WARN 헬스체크 실패: {e}')

print('\n이 URL 을 본 채팅에 알려주세요 (프론트엔드 .env 의 NEXT_PUBLIC_API_URL 에 설정):')
print(f'   {base_url}')